In [1]:
import gpboost as gpb
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
# Load data
data = pd.read_csv('C:/Files/GLAN_DATA/first_wave/downscale_data_60/data_with_activity_sequence_all_recod_downsampling_activity_travel.csv')
data = data[data['time_weight'] >= 5]
data = data[data['tree_height'] >= 0]
print(data.shape)
pred_vars = ['household_income', 'age', 'gender', 'education_level', 'employment_status', 'neighborhood_type',
             'work_study', 'housework', 'personal_affair', 'leisure',
             'travel', 'transportation', 'residence', 'industry', 'company', 'shopping',
             'restaurant', 'life_service', 'education_culture', 'entertainment', 'sport_fitness',
             'recreation_tourism', 'healthcare', 'workday', 'time_hour', 'mobility_status',
             'time_nonflexibility', 'home_ornot', 'POI_density', 'POI_diversity', "tree_height"]
# Prepare 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=666)
fold_results = []


(8427, 52)


C:\anaconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import optuna
from optuna.samplers import TPESampler
import gpboost as gpb
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold

# Define the Optuna objective function (to be used per fold)
def create_objective(data_train_fold, data_val_fold):
    def objective(trial):
        params = {
            'num_boost_round': trial.suggest_int('num_boost_round', 200, 500),
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'num_leaves': trial.suggest_int('num_leaves', 12, 200),
            'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
            'lambda_l1': trial.suggest_float('lambda_l1', 1, 50, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1, 100, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
            'min_gain_to_split': trial.suggest_float('min_gain_to_split', 1, 5),
            'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1, 10, log=True),
            'verbose': 0,
            'objective': 'regression',
            'metric': 'mse'
        }

        
        gp_model = gpb.GPModel(group_data=data_train_fold['pid'], likelihood='gaussian')
        data_bst = gpb.Dataset(data=data_train_fold[pred_vars], label=data_train_fold['compound_exposure_disadvantage'])

        
        data_bst_valid = gpb.Dataset(data=data_val_fold[pred_vars], label=data_val_fold['compound_exposure_disadvantage'], reference=data_bst)

        
        gpbst = gpb.train(
            params=params,
            train_set=data_bst,
            gp_model=gp_model
        )

        
        pred_val = gpbst.predict(data=data_val_fold[pred_vars], group_data_pred=data_val_fold['pid'], pred_latent=False)['response_mean']

        
        mse = mean_squared_error(data_val_fold['compound_exposure_disadvantage'], pred_val)
        r2 = r2_score(data_val_fold['compound_exposure_disadvantage'], pred_val)
        print(f"Trial {trial.number} - MSE: {mse:.4f}, R²: {r2:.4f}")

        return r2

    return objective

# Perform optimization for each fold
for fold, (train_idx, val_idx) in enumerate(kf.split(data)):
    print(f"Starting optimization for fold {fold + 1}")
    data_train_fold = data.iloc[train_idx]
    data_val_fold = data.iloc[val_idx]

    
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=666), study_name=f'optimization_fold_{fold + 1}')
    objective = create_objective(data_train_fold, data_val_fold)
    study.optimize(objective, n_trials=50)

    # Get best parameters and best R² (opt_val_r2)
    best_params = study.best_params
    opt_val_r2 = study.best_value

    
    final_params = best_params.copy()
    final_params.update({'verbose': 0, 'objective': 'regression', 'metric': 'mse'})

    gp_model = gpb.GPModel(group_data=data_train_fold['pid'], likelihood='gaussian')
    data_bst = gpb.Dataset(data=data_train_fold[pred_vars], label=data_train_fold['compound_exposure_disadvantage'])
    data_bst_valid = gpb.Dataset(data=data_val_fold[pred_vars], label=data_val_fold['compound_exposure_disadvantage'], reference=data_bst)

    gpbst = gpb.train(
        params=final_params,
        train_set=data_bst,
        gp_model=gp_model
    )

    
    pred_train = gpbst.predict(data=data_train_fold[pred_vars], group_data_pred=data_train_fold['pid'], pred_latent=False)['response_mean']
    train_mse = mean_squared_error(data_train_fold['compound_exposure_disadvantage'], pred_train)
    train_r2 = r2_score(data_train_fold['compound_exposure_disadvantage'], pred_train)

    
    pred_val = gpbst.predict(data=data_val_fold[pred_vars], group_data_pred=data_val_fold['pid'], pred_latent=False)['response_mean']
    val_mse = mean_squared_error(data_val_fold['compound_exposure_disadvantage'], pred_val)
    val_r2 = r2_score(data_val_fold['compound_exposure_disadvantage'], pred_val)

    
    fold_results.append({
        'fold': fold + 1,
        'best_params': best_params,
        'train_mse': train_mse,
        'train_r2': train_r2,
        'val_mse': val_mse,
        'val_r2': val_r2
    })

    print(f"Fold {fold + 1} - Best Params: {best_params}")
    print(f"Fold {fold + 1} - Train MSE: {train_mse:.4f}")
    print(f"Fold {fold + 1} - Train R²: {train_r2:.4f}")
    print(f"Fold {fold + 1} - Val MSE: {val_mse:.4f}")
    print(f"Fold {fold + 1} - Val R²: {val_r2:.4f}")


print("All fold results:")
for result in fold_results:
    print(result)

[I 2025-10-04 15:34:41,327] A new study created in memory with name: optimization_fold_1


Starting optimization for fold 1


C:\anaconda3\envs\torch\lib\site-packages\gpboost\engine.py:183: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))
[I 2025-10-04 15:34:41,907] Trial 0 finished with value: -12.981249702830459 and parameters: {'num_boost_round': 410, 'learning_rate': 0.08759892712381777, 'max_depth': 10, 'num_leaves': 149, 'min_data_in_leaf': 191, 'lambda_l1': 1.05095071676774, 'lambda_l2': 6.717000762100074, 'feature_fraction': 0.61464383814, 'min_gain_to_split': 1.3997142452848457, 'min_sum_hessian_in_leaf': 3.22156060444316}. Best is trial 0 with value: -12.981249702830459.


Trial 0 - MSE: 0.6602, R²: -12.9812


C:\anaconda3\envs\torch\lib\site-packages\gpboost\engine.py:183: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))
[I 2025-10-04 15:34:42,174] Trial 1 finished with value: -3.170717179812878 and parameters: {'num_boost_round': 260, 'learning_rate': 0.05156096819997906, 'max_depth': 5, 'num_leaves': 144, 'min_data_in_leaf': 66, 'lambda_l1': 20.69270954475357, 'lambda_l2': 1.0238060168070713, 'feature_fraction': 0.6338572960895124, 'min_gain_to_split': 1.4438146879497866, 'min_sum_hessian_in_leaf': 1.7687572324193976}. Best is trial 1 with value: -3.170717179812878.


...

Trial 48 - MSE: 0.2990, R²: -5.3384


C:\anaconda3\envs\torch\lib\site-packages\gpboost\engine.py:183: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))
[I 2025-10-04 15:36:13,952] Trial 49 finished with value: 0.1186418471470484 and parameters: {'num_boost_round': 276, 'learning_rate': 0.010847962411650765, 'max_depth': 5, 'num_leaves': 97, 'min_data_in_leaf': 94, 'lambda_l1': 49.72789165779987, 'lambda_l2': 17.0405762226387, 'feature_fraction': 0.7686842391205024, 'min_gain_to_split': 4.983212993935939, 'min_sum_hessian_in_leaf': 7.775356140088421}. Best is trial 17 with value: 0.712401890783825.


Trial 49 - MSE: 0.0416, R²: 0.1186


C:\anaconda3\envs\torch\lib\site-packages\gpboost\engine.py:183: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))


Fold 5 - Best Params: {'num_boost_round': 340, 'learning_rate': 0.0015894006086745686, 'max_depth': 9, 'num_leaves': 12, 'min_data_in_leaf': 23, 'lambda_l1': 29.059077924732936, 'lambda_l2': 20.072091736120427, 'feature_fraction': 0.7434875033909883, 'min_gain_to_split': 2.027129188502712, 'min_sum_hessian_in_leaf': 4.774274764398325}
Fold 5 - Train MSE: 0.0055
Fold 5 - Train R²: 0.8854
Fold 5 - Val MSE: 0.0136
Fold 5 - Val R²: 0.7124
All fold results:
{'fold': 1, 'best_params': {'num_boost_round': 327, 'learning_rate': 0.0010736644927379778, 'max_depth': 8, 'num_leaves': 28, 'min_data_in_leaf': 123, 'lambda_l1': 38.24015706443186, 'lambda_l2': 58.402742781712, 'feature_fraction': 0.7869551121170515, 'min_gain_to_split': 2.2310470435716, 'min_sum_hessian_in_leaf': 5.435428232968467}, 'train_mse': 0.005850899414546548, 'train_r2': 0.8773753964489281, 'val_mse': 0.013366758652394591, 'val_r2': 0.7169492355568612}
{'fold': 2, 'best_params': {'num_boost_round': 280, 'learning_rate': 0.0012

In [2]:
import gpboost as gpb
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

#, log=True
best_params = {
    'num_boost_round': 280,
    'learning_rate': 0.001292112722051777,
    'max_depth': 5,
    'num_leaves': 127,
    'min_data_in_leaf': 45,
    'lambda_l1': 22.605341824915648,
    'lambda_l2': 7.170332586173781,
    'feature_fraction': 0.7386048051720877,
    'min_gain_to_split': 4.374371803180026,
    'min_sum_hessian_in_leaf': 1.7580285746703073
}

final_params = best_params.copy()
final_params.update({'verbose': -1, 'objective': 'regression', 'metric': 'mse'})



gp_model = gpb.GPModel(group_data=data['pid'], likelihood='gaussian')
data_bst = gpb.Dataset(data=data[pred_vars], label=data['compound_exposure_disadvantage'])
gpbst = gpb.train(
    params=final_params,
    train_set=data_bst,
    gp_model=gp_model
)



pred_latent = gpbst.predict(
    data=data[pred_vars],
    group_data_pred=data['pid'],
    pred_latent=True
)
pred_fixed = pred_latent['fixed_effect']
pred_random = pred_latent['random_effect_mean']
pred_full = pred_fixed + pred_random

C:\anaconda3\envs\torch\lib\site-packages\gpboost\engine.py:183: UserWarning: Found `num_boost_round` in params. Will use it instead of argument
  _log_warning("Found `{}` in params. Will use it instead of argument".format(alias))


In [3]:
gp_model.summary()
var_error_term=gp_model.get_cov_pars()["Error_term"]
var_random_effects=gp_model.get_cov_pars()["pid"]
var_fixed_effects=np.var(pred_fixed)
icc= var_random_effects/(var_random_effects+var_error_term)
marginal_r2= var_fixed_effects/(var_error_term+var_random_effects+var_fixed_effects)
condition_r2=(var_fixed_effects+var_random_effects)/(var_error_term+var_random_effects+var_fixed_effects)
print(f"var_fixed_effects:{var_fixed_effects}")
print(f"ICC: {icc}")
print(f"Marginal R²: {marginal_r2}")
print(f"Condition R²: {condition_r2}")

Model summary:
Nb. observations: 8427
Nb. groups: 749 (pid)
Covariance parameters (random effects):
            Param.
Error_term  0.0060
pid         0.0043
var_fixed_effects:0.03142662599965965
ICC: Param.    0.419414
dtype: float64
Marginal R²: Param.    0.751968
dtype: float64
Condition R²: Param.    0.855996
dtype: float64
